**Importing the necessary libraries**

In [ ]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import random
import torch
import torch.nn.functional as F

In [ ]:
# Encoding Can you help me to translate this sentence
st='Can you help me to translate this sentence'
r=np.arange(0,len(st.split()))
np.random.shuffle(r)
print(type(r))
dic={w:n for w,n in zip(st.split(),r)}

<class 'numpy.ndarray'>


In [ ]:
# Creating the torch tensor
sentence=torch.tensor([5,1,0,2,4,3,7,6])

In [ ]:
sentence

tensor([5, 1, 0, 2, 4, 3, 7, 6])

In [ ]:
# Creating an embedding
torch.manual_seed(11)
embed=nn.Embedding(10,16)
embedded_sentence=embed(sentence).detach()
embedded_sentence.shape

torch.Size([8, 16])

In [ ]:
# Computing omega as dot product between i,j embedding
omega=torch.empty(8,8)
for i,x_i in enumerate(embedded_sentence):
  for j,x_j in enumerate(embedded_sentence):
    omega[i,j]=torch.dot(x_i,x_j)


In [ ]:
# Matrix multiplication
omegamat=embedded_sentence.matmul(embedded_sentence.T)

In [ ]:
# Comparing two matrices
torch.allclose(omega,omegamat)

True

In [ ]:
# Normalizing weights using softmax function
attention_weights=F.softmax(omegamat,dim=1)
attention_weights.shape

torch.Size([8, 8])

In [ ]:
# Attention weights sum to 1 along each row
attention_weights.sum(dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [ ]:
# Computing context vectors for the second raw
x2=embedded_sentence[1,:]
context_vec_2=torch.zeros(x2.shape)
for j in range(8):
  x_j=embedded_sentence[j,:]
  context_vec_2+=attention_weights[1,j]*x_j

# Printing the resulting vector
context_vec_2

NameError: name 'attention_weights' is not defined

In [ ]:
# Achieving the same result using matrix multiplication
context_vectors=torch.matmul(attention_weights,embedded_sentence)

In [ ]:
# Comparing the results
torch.allclose(context_vectors[1],context_vec_2)

True

In [ ]:
# Creating matrices
torch.manual_seed(88)
d=embedded_sentence.shape[1]
U_query=torch.randn(d,d)
U_key=torch.randn(d,d)
U_val=torch.randn(d,d)

In [ ]:
# Projecting the query sequence
x_2=embedded_sentence[1]
query_2=x_2.matmul(U_query)
query_2

tensor([ 3.7221,  7.6859, -6.2859,  2.5911,  1.3357, -5.5493, -1.4358, -1.7758,
         4.7922,  2.7718,  4.4691, -3.1209, -2.4210, -2.5499, -2.0691,  4.8180])

In [ ]:
# Computing key and value sentences
key_2=U_key.matmul(x_2)
val_2=U_val.matmul(x_2)

In [ ]:
# Computing the key and value matrices
keys=U_key.matmul(embedded_sentence.T).T
vals=U_val.matmul(embedded_sentence.T).T
torch.allclose(key_2,keys[1])
torch.allclose(val_2,vals[1])

True

In [ ]:
# Calculating weights 23 and weights between second elements and other elements
omega23=query_2.dot(keys[2])
omega2=query_2.matmul(keys.T)

In [ ]:
# Normalizing dividing through d_k**0.5 and applying softmax function
attention_weights_2=F.softmax(omega2/(d**0.5))

/tmp/ipykernel_2581/1940308643.py:2: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  attention_weights_2=F.softmax(omega2/(d**0.5))


In [ ]:
# Finally multiplying nomralized weights with values
context_vec_2=attention_weights_2.matmul(vals)
context_vec_2

tensor([-3.1256, -1.6897,  3.1429, -3.5132,  2.9262, -1.4674, -3.0523,  6.3993,
        -6.3072,  6.9956, -3.3653, -2.3751,  1.6711,  0.4039, -5.1571, -5.1594])

# Multihead Attention

In [ ]:
# Creating one query matrix
torch.manual_seed(11)
d=embedded_sentence.shape[1]
one_U_query=torch.rand(d,d)

In [ ]:
# Adding multiple heads by adding additional dimension
h=8
multiple_U_query=torch.rand(h,d,d)
multiple_U_key=torch.rand(h,d,d)
multiple_U_value=torch.rand(h,d,d)


In [ ]:
# The multihead projections of the second word
x_2=embedded_sentence[1]
multihead_query_2=multiple_U_query.matmul(x_2)
multihead_query_2.shape

torch.Size([8, 16])

In [ ]:
# Conducting the multiplication
multihead_key_2=multiple_U_key.matmul(x_2)
multihead_value_2=multiple_U_value.matmul(x_2)
multihead_key_2[3]

tensor([-0.3014,  0.6397,  0.5385, -1.0113,  1.7586, -0.1405, -1.3345, -0.0582,
        -0.2146, -1.3265,  1.9245,  3.5436,  2.6436,  4.0650,  0.7382,  1.2348])

In [ ]:
# Repaeting the tensor along one of the dimensions
stacked_inputs=embedded_sentence.T.repeat(8,1,1)
stacked_inputs.shape

torch.Size([8, 16, 8])

In [ ]:
# Batch matrix multiplication
multihead_keys=torch.bmm(multiple_U_key,stacked_inputs)
multihead_keys.shape

torch.Size([8, 16, 8])

In [ ]:
# Permuting the keys
multihead_keys=multihead_keys.permute(0,2,1)
multihead_keys.shape

torch.Size([8, 8, 16])

In [ ]:
# Retrieving the second value in the second attention layer
multihead_keys[2,1]

tensor([-0.9617, -0.1051,  1.2500, -0.0451,  0.3779, -1.3755,  0.5477,  0.6317,
         1.7153,  0.4511,  1.0004,  0.0136,  0.1188,  0.3795, -1.4176,  0.5415])

In [ ]:
# Repeating this for the value sequence
multihead_values=torch.matmul(multiple_U_value,stacked_inputs)
multihead_values=multihead_values.permute(0,2,1)
multihead_values.shape

torch.Size([8, 8, 16])

In [ ]:
# Context vectors for the second element
multihead_z_2=torch.rand(8,16)

In [ ]:
# Employing concatenation and squashing
linear=nn.Linear(8*16,16)
context_vector_2=linear(multihead_z_2.flatten())

# Creating GPT-2

In [ ]:
!pip install transformers

In [ ]:
from transformers import pipeline, set_seed
from accelerate import Accelerator

In [ ]:
generator=pipeline(task='text-generation',model='gpt2')

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
set_seed(111)
generator('Hey, reader, today is',max_length=30,num_return_sequences=4)

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': "Hey, reader, today is your day off. We're going to talk about you, the guy who wrote the original game. You're our hero; you're a giant character in the game! We'll talk about all kinds of things.\n\nLet's start with this: Before he was a human being, you didn't really exist. You were the hero. After you died, he was the hero. And you were the hero. And now, you've been a hero for a long time. And now you're a hero for life.\n\nYou're a giant in the game. You're a human being in the game, and you're a superhero. The game puts you in a situation where you can't fight it and you have to fight it with dignity. And then you're the hero.\n\nAnd then, you're the hero. And now you're the hero.\n\nSo is it a good idea to have a hero? Absolutely not. It's not a bad idea to have a hero. In fact it's a bad idea to have a hero who's been around for a long time. And you're the hero, and you're the hero is the hero.\n\nIt's a good idea to have a hero who's been around for a"},
 

In [ ]:
# Using Tokenizer from transformer model
from transformers import GPT2Tokenizer
tokenizer=GPT2Tokenizer.from_pretrained('gpt2')
text=torch.tensor('Let us encode this sentence')
encoded_input=tokenizer(text,return_tensor='pt')
encoded_input

In [ ]:
from transformers import GPT2Model
import torch # Ensure torch is imported for tensor operations

model=GPT2Model.from_pretrained('gpt2')

# Ensure encoded_input values are tensors and have a batch dimension
if isinstance(encoded_input['input_ids'], list):
    encoded_input['input_ids'] = torch.tensor(encoded_input['input_ids']).unsqueeze(0)
if isinstance(encoded_input['attention_mask'], list):
    encoded_input['attention_mask'] = torch.tensor(encoded_input['attention_mask']).unsqueeze(0)

# GPT2 based feature encodings
output=model(**encoded_input)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
output['last_hidden_state'].shape

torch.Size([1, 5, 768])

# Building BERT model

In [ ]:
!pip install transformers
!pip install torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 5.8 MB/s eta 0:00:00


In [ ]:
! pip show transformers

Name: transformers
Version: 5.16.1
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.13/dist-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: peft, sentence-transformers


In [ ]:
import gzip
import shutil
import time
import pandas as pd
import requests
import torch.nn.functional as F
import torch
import torchtext
import transformers
from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification
from pathlib import Path

In [ ]:
# Specifying general settings
torch.backends.cudnn.determenistic=True
random_seed=123
torch.manual_seed(random_seed)

In [ ]:
# Specifying the device
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs=3

In [ ]:
# Reading the file of interest
name=r"/content/movie_data.csv.gz"
fil_path=Path(name)
file_name=fil_path.stem
print(file_name)

movie_data.csv


In [ ]:
# Reading the zipped file and writing new .csv file
with gzip.open(fil_path,'rb') as fil_out:
  with open('movie_data.csv','wb') as fil_in:
    shutil.copyfileobj(fil_out,fil_in)

In [ ]:
# Reading the movie as pandas df
df=pd.read_csv('movie_data.csv')
df.head()

,review,sentiment
0,"In 1974, the teenager Martha Moxley (Maggie Gr...",1
1,OK... so... I really like Kris Kristofferson a...,0
2,"***SPOILER*** Do not read this, if you think a...",0
3,hi for all the people who have seen this wonde...,1
4,"I recently bought the DVD, forgetting just how...",0


In [ ]:
# Splitting traing,test and validation text and responses
train_texts=df.iloc[:35000,:]['review'].values
train_labels=df.iloc[:35000,:]['sentiment'].values

test_texts=df.iloc[35000:40000,:]['review'].values
test_labels=df.iloc[35000:40000,:]['sentiment'].values

valid_texts=df.iloc[40000:,:]['review'].values
valid_labels=df.iloc[40000:,:]['sentiment'].values



# Tokenizing the text

In [ ]:
# Using from_pretrained to load pretrained artifacts
tokenizer=DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
# Making encodings with the aid of tokenizer
train_encodings=tokenizer(list(train_texts),padding=True,truncation=True)
test_encodings=tokenizer(list(test_texts),padding=True,truncation=True)
valid_encodings=tokenizer(list(valid_texts),padding=True,truncation=True)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
# Creating a custom dataset
from torch.utils.data import Dataset

In [ ]:
class IMBdDataset(Dataset):
  def __init__(self,encodings,labels):
    self.encodings=encodings
    self.labels=labels
  # Defining two custom methods len() and getitem()
  def __getitem__(self,idx):
    item={key:torch.tensor(val[idx]) for key,val in self.encodings.items()}
    item['labels']=torch.tensor(self.labels[idx])
    return item
  def __len__(self):
    return len(self.labels)



In [ ]:
# Creating datasets and dataloaders
train_dataset=IMBdDataset(train_encodings,train_labels)
test_dataset=IMBdDataset(test_encodings,test_labels)
valid_dataset=IMBdDataset(valid_encodings,valid_labels)

In [ ]:
# Creating dataloaders
from torch.utils.data import DataLoader
train_loader=DataLoader(train_dataset,batch_size=16,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=16,shuffle=False)
valid_loader=DataLoader(valid_dataset,batch_size=16,shuffle=False)

In [ ]:
# Downloading and finetuning the model
model=DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Device specification
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
# Using the Adam optimizer
optim=torch.optim.Adam(model.parameters(),lr=5e-5)

In [ ]:
# Defining the function for accuracy computation
def compute_accuracy(model,data_loader,device):
  with torch.no_grad():
    correct_pred,num_examples=0,0
    for batch_idx,batch in enumerate(data_loader):
      # Preparing the data
      input_ids=batch['input_ids'].to(device)
      attention_mask=batch['attention_mask'].to(device)
      labels=batch['labels'].to(device)
      model.train()
      # Conducting the prediction based on the model
      outputs=model(input_ids=input_ids,attention_mask=attention_mask)
      logits=outputs['logits']
      predicted_labels=torch.argmax(logits,1)
      num_examples+=(labels.size(0))
      correct_pred+=(predicted_labels==labels).sum()
    return correct_pred.float()/num_examples * 100

In [ ]:
# Finetuning the transformer architecture
start_time=time.time()
num_epochs=3
for epoch in range(num_epochs):
  model.train()
  for batch_idx,batch in enumerate(train_loader):
    # Preparing the data
    input_ids=batch['input_ids'].to(device)
    attention_mask=batch['attention_mask'].to(device)
    labels=batch['labels'].to(device)
    # Forward pass
    outputs=model(input_ids=input_ids,attention_mask=attention_mask,labels=labels)
    loss,logits=outputs['loss'],outputs['logits']

    # Backward pass
    optim.zero_grad()
    loss.backward()
    optim.step()

    # Logging
    if not batch_idx % 100:
      print(f'Epoch: {epoch+1} | Batch: {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')

    # Conducting the evaluation
  model.eval()
  with torch.set_grad_enabled(False):
    print(f'Training Accuracy: {compute_accuracy(model,train_loader,device):.3f}%, \n Validation Accuracy {compute_accuracy(model,valid_loader,device):.3f}%')
    print(f'Time elapsed {(time.time()-start_time)/60 :.2f}')
  print(f'Total Training Time {(time.time()-start_time)/60 :.2f} min')
  print(f'Test Accuracy {compute_accuracy(model,test_loader,device):.3f}%')



In [ ]:
# Using the Trainer API from transformer library
model=DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Specifying the device and moving our model into it
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [ ]:
# Specifying the optimizer
optim=torch.optim.Adam(model.parameters(),lr=10e-5)

In [ ]:
# Importing the Trainer API to substitute the boilerplate code
from transformers  import Trainer, TrainingArguments


In [ ]:
# Generating Training Arguments as well as Trainer
training_argd=TrainingArguments(output_dir='./results',num_train_epochs=3,per_device_train_batch_size=16,per_device_eval_batch_size=16,log_level='info',logging_steps=10)
# trainer=Trainer(model=model,args=training_argd,train_dataset=train_dataset,optimizers=(optim,None))


In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00


In [ ]:
# Import the load metrics
import evaluate
import numpy as np

metric=evaluate.load("accuracy")

# Defining the function for metric computations
def compute_met(eval_pred):
  logits,labels=eval_pred
  predictions=np.argmax(logits,axis=1)
  return metric.compute(predictions=predictions,references=labels)


In [ ]:
# Training the model
trainer=Trainer(model=model,args=training_argd,train_dataset=train_dataset,optimizers=(optim,None),compute_metrics=compute_met)
start_time=time.time()
trainer.train() # train method to employ the trainer

[transformers] ***** Running training *****
[transformers]   Num examples = 35,000
[transformers]   Num Epochs = 3
[transformers]   Num update steps per epoch = 2,188
[transformers]   Instantaneous batch size per device = 16
[transformers]   Total train batch size (w. parallel, distributed & accumulation) = 16
[transformers]   Gradient Accumulation steps = 1
[transformers]   Total optimization steps = 6,564
[transformers]   Number of trainable parameters = 66,955,010


Step,Training Loss
10,0.696754
20,0.649636
30,0.477969
40,0.386231
50,0.359018
60,0.339828
70,0.367629
80,0.440741
90,0.340917
100,0.254533


[transformers] Saving model checkpoint to ./results/checkpoint-500
[transformers] Configuration saved in ./results/checkpoint-500/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in ./results/checkpoint-500/model.safetensors
[transformers] Saving model checkpoint to ./results/checkpoint-1000
[transformers] Configuration saved in ./results/checkpoint-1000/config.json


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in ./results/checkpoint-6500/model.safetensors
[transformers] Saving model checkpoint to ./results/checkpoint-6564
[transformers] Configuration saved in ./results/checkpoint-6564/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Model weights saved in ./results/checkpoint-6564/model.safetensors
[transformers] 

Training completed. Do not forget to share your model on huggingface.co/models =)




TrainOutput(global_step=6564, training_loss=0.15564811795309758, metrics={'train_runtime': 5408.8897, 'train_samples_per_second': 19.412, 'train_steps_per_second': 1.214, 'total_flos': 1.390907685888e+16, 'train_loss': 0.15564811795309758, 'epoch': 3.0})